In [1]:
import warnings
from pathlib import Path
from collections import Counter

import scipy as sp
import numpy as np
import pandas as pd
from rectools import Columns
from rectools.metrics import MAP, MeanInvUserFreq
from rectools.dataset import Dataset
from rectools.models.popular_in_category import PopularInCategoryModel
from rectools.models.popular import PopularModel
from scipy.sparse import coo_matrix, spmatrix
from implicit.nearest_neighbours import ItemItemRecommender
from tqdm.auto import tqdm


warnings.filterwarnings("ignore")

In [2]:
data_base_path = Path("../data/raw/kion_train/").resolve()

interactions = pd.read_csv(data_base_path / "interactions.csv", parse_dates=["last_watch_dt"])
interactions = interactions.rename(columns={"total_dur": Columns.Weight, "last_watch_dt": Columns.Datetime})

users = pd.read_csv(data_base_path / "users.csv")
items = pd.read_csv(data_base_path / "items.csv")

print(interactions.shape)
interactions.head(5)

(5476251, 5)


,user_id,item_id,datetime,weight,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


In [3]:
class UserKnn:
    SIMILAR_USER_COLUMN = "similar_user_id"
    SIMILARITY_COLUMN = "similarity"
    IDF_COLUMN = "idf"

    def __init__(
        self,
        model: ItemItemRecommender,
        pop_model: PopularModel,
        pop_cat_model: PopularInCategoryModel,
        N_similar_users: int,
    ):
        self.model = model
        self.N_similar_users = N_similar_users

        self.users_inv_mapping = None
        self.users_mapping = None
        self.items_inv_mapping = None
        self.items_mapping = None

        self.watched_items_dataframe = None
        self.item_idf = None
        self.cold_model_fitted = False
        self.cold_items = None
        
        self.pop_model = pop_model
        self.dataset_pop = None
        self.cold_model_fitted = False

        self.pop_cat_model = pop_cat_model
        self.items = items.copy()
        self.dataset_feature = None

    def _set_mappings(self, interactions: pd.DataFrame) -> None:
        """
        Create dictionaries to map external IDs (users, items) to internal IDs and vice versa.
        """
        unique_users = interactions[Columns.User].unique()
        self.users_inv_mapping = dict(enumerate(unique_users))
        self.users_mapping = {v: k for k, v in self.users_inv_mapping.items()}

        unique_items = interactions[Columns.Item].unique()
        self.items_inv_mapping = dict(enumerate(unique_items))
        self.items_mapping = {v: k for k, v in self.items_inv_mapping.items()}

    def _get_user_item_matrix(self, interactions: pd.DataFrame) -> spmatrix:
        """
        Construct a sparse user-item matrix in CSR format.
        Rows represent users, and columns represent items.
        """
        user_idx = interactions[Columns.User].map(self.users_mapping.get)
        item_idx = interactions[Columns.Item].map(self.items_mapping.get)
        data = interactions[Columns.Weight].astype(np.float64)

        user_item_coo = coo_matrix((data, (user_idx, item_idx)))
        return user_item_coo.tocsr()

    def _set_interacted_items_dataframe(self, interactions: pd.DataFrame) -> None:
        """
        Groups interactions by user to get item_id list for each user
        """
        self.interacted_items_dataframe = (
            interactions.groupby(Columns.User, as_index=False)
            .agg({Columns.Item: list})
            .rename(columns={Columns.User: self.SIMILAR_USER_COLUMN})
        )

    @staticmethod
    def idf(n: int, x: float):
        """
        Calculates IDF for one item
        """
        return np.log((1 + n) / (1 + x) + 1)

    def _count_item_idf(self, interactions: pd.DataFrame) -> None:
        """
        Calculate IDF values for all items present in the interactions dataset
         and store the result in self.item_idf.
        """
        item_freqs = Counter(interactions[Columns.Item].values)
        item_idf_df = pd.DataFrame.from_dict(item_freqs, orient="index", columns=["doc_freq"]).reset_index()
        total_interactions = len(interactions)
        item_idf_df[self.IDF_COLUMN] = item_idf_df["doc_freq"].apply(lambda x: self.idf(total_interactions, x))
        self.item_idf = item_idf_df

    def _prepare_for_model(self, train_interactions: pd.DataFrame) -> None:
        """
        Sets mappings, grouped interactions, calculates idf
        """
        self._set_mappings(train_interactions)
        self._set_interacted_items_dataframe(train_interactions)
        self._count_item_idf(train_interactions)

    def fit_cold_model(self, train_interactions: pd.DataFrame) -> None:
        self.dataset_pop = Dataset.construct(
            interactions_df=train_interactions,
            user_features_df=None,
            item_features_df=None
        )
        self.pop_model.fit(self.dataset_pop)
        self.cold_model_fitted = True

    def recommend_cold(self, users: list, k: int = 100) -> pd.DataFrame:
        if not self.cold_model_fitted:
            raise ValueError("Cold model not fitted. Call fit_cold_model first.")
        
        recs = self.pop_model.recommend(users, dataset=self.dataset_pop, k=k, filter_viewed=False)
        return recs

    def _set_dataset_feature(self, train_interactions: pd.DataFrame) -> None:
        self.items["genre"] = self.items["genres"].str.split(", ")
        genre_feature = self.items[["item_id", "genre"]].explode("genre")
        genre_feature.columns = ["id", "value"]
        genre_feature["feature"] = "genre"

        valid_items = train_interactions[Columns.Item].unique()
        genre_feature = genre_feature[genre_feature["id"].isin(valid_items)]

        self.dataset_feature = Dataset.construct(
            interactions_df=train_interactions,
            user_features_df=None,
            item_features_df=genre_feature,
            cat_item_features=["genre"],
        )

    def fit(self, train_interactions: pd.DataFrame) -> None:
        """
        Fit the model on the provided training data.

        Internally:
        1) Prepare mappings, watchlist DataFrame, and item IDF.
        2) Create a user-item matrix and fit the underlying Implicit model.
        """
        self._prepare_for_model(train_interactions)
        user_item_matrix = self._get_user_item_matrix(train_interactions)

        self._set_dataset_feature(train_interactions)
        self.pop_cat_model.fit(self.dataset_feature)

        self.model.fit(user_item_matrix.T)
        self.fit_cold_model(train_interactions)

    def _get_similar_users(self, external_user_id: int) -> tuple[list[int], list[float]]:
        """
        Retrieve a list of similar users and corresponding similarities
        from the underlying Implicit model.
        """
        if external_user_id not in self.users_mapping:
            # if user doesn't exist in mapping, return sentinel (-1).
            return [-1], [-1]

        internal_user_id = self.users_mapping[external_user_id]
        user_ids, similarities = self.model.similar_items(internal_user_id, N=self.N_similar_users)
        # convert back to external IDs
        external_user_ids = [self.users_inv_mapping[u_id] for u_id in user_ids]
        return external_user_ids, similarities

    @staticmethod
    def get_rank(recs: pd.DataFrame, k: int) -> pd.DataFrame:
        """
        Sort recommendations by score in descending order,
        assign ranks within each user group, and then truncate by top-k.
        """
        recs = recs.sort_values([Columns.User, Columns.Score], ascending=False)
        recs = recs.drop_duplicates([Columns.User, Columns.Item])
        recs[Columns.Rank] = recs.groupby(Columns.User).cumcount() + 1
        recs = recs[recs[Columns.Rank] <= k][[Columns.User, Columns.Item, Columns.Score, Columns.Rank]]

        return recs

    def recommend(self, users: np.ndarray, k: int) -> pd.DataFrame:
        """
        Generate recommendations for each user by combining:
        - Hot recs from the ItemItem model,
        - 10 cold recs from global popularity, and
        - 10 category recs from the popular-by-category model.

        The final ranking is done by get_rank.
        """
        recs_hot = pd.DataFrame({Columns.User: users})
        recs_hot[self.SIMILAR_USER_COLUMN], recs_hot[self.SIMILARITY_COLUMN] = zip(
            *recs_hot[Columns.User].map(lambda user_id: self._get_similar_users(user_id))
        )
        recs_hot = recs_hot.set_index(Columns.User).apply(pd.Series.explode).reset_index()

        recs_hot = (
            recs_hot[~(recs_hot[Columns.User] == recs_hot[self.SIMILAR_USER_COLUMN])]
            .merge(self.interacted_items_dataframe, on=[self.SIMILAR_USER_COLUMN], how="left")
            .explode(Columns.Item)
            .sort_values([Columns.User, self.SIMILARITY_COLUMN], ascending=False)
            .drop_duplicates([Columns.User, Columns.Item], keep="first")
            .merge(self.item_idf, left_on=Columns.Item, right_on="index", how="left")
        )
        recs_hot[Columns.Score] = recs_hot[self.SIMILARITY_COLUMN] * recs_hot[self.IDF_COLUMN]
        recs_hot = recs_hot[[Columns.User, Columns.Item, Columns.Score]]
        print("Hot recommendations done (step 1)")

        if not hasattr(self, "_cached_cold_recs"):
            dummy_cold = self.recommend_cold(users=[-1], k=10)
            self._cached_cold_recs = dummy_cold[[Columns.Item, Columns.Score]].copy()

        recs_cold = pd.concat([self._cached_cold_recs.assign(**{Columns.User: u}) for u in users], ignore_index=True)
        print("Cold recommendations done (step 2)")

        if not hasattr(self, "_cached_cat_recs"):
            dummy_cat = self.pop_cat_model.recommend([-1], dataset=self.dataset_feature, k=10, filter_viewed=False)
            self._cached_cat_recs = dummy_cat[[Columns.Item, Columns.Score]].copy()

        recs_cat = pd.concat([self._cached_cat_recs.assign(**{Columns.User: u}) for u in users], ignore_index=True)
        print("Category recommendations done (step 3)")

        recs_all = pd.concat([recs_hot, recs_cold, recs_cat], ignore_index=True)
        return self.get_rank(recs_all, k=k)

In [4]:
user_knn = UserKnn(
    model=ItemItemRecommender(),
    pop_model=PopularModel(),
    pop_cat_model=PopularInCategoryModel(category_feature="genre", n_categories=5),
    N_similar_users=10,
)

In [5]:
user_knn.fit(interactions)

  0%|          | 0/962179 [00:00<?, ?it/s]

In [6]:
import pickle

models_base_path = Path("../models/").resolve()

with open(models_base_path / "user_knn_model_whole.pkl", "wb") as f:
    pickle.dump(user_knn, f)

with open(models_base_path / "user_knn_model_whole.pkl", "rb") as f:
    user_knn = pickle.load(f)

In [7]:
hot_users = interactions["user_id"].unique()
print("Number of hot users:", len(hot_users))

Number of hot users: 962179


In [8]:
reco_hot = user_knn.recommend(hot_users, k=10)
reco_hot = reco_hot[[Columns.User, Columns.Item, Columns.Rank]].rename(
    columns={Columns.User: "user_id", Columns.Item: "item_id", Columns.Rank: "rank"}
)
print("Hot recommendations (sample):")
print(reco_hot.head())

Hot recommendations done (step 1)
Cold recommendations done (step 2)
Category recommendations done (step 3)
Hot recommendations (sample):
    user_id item_id  rank
2   1097557    3472     1
21  1097557    9449     2
33  1097557    1582     3
16  1097557    4930     4
23  1097557   14624     5


In [9]:
reco_hot.to_csv(data_base_path / "reco_hot_whole_7.csv", index=False)

In [10]:
data_base_path = Path("../data/raw/kion_train/").resolve()

In [3]:
reco_hot = pd.read_csv(data_base_path / "reco_hot_whole.csv")

In [12]:
reco_hot_sorted = reco_hot.sort_values("rank")

In [17]:
reco_hot_sorted[reco_hot_sorted["user_id"] == 1]

,user_id,item_id,rank
135940262,1,14603,1
135940264,1,7713,2
135940263,1,6006,3
135940261,1,9996,4
135940284,1,11238,5
135940281,1,3567,6
135940268,1,7570,7
135940269,1,6907,8
135940278,1,2028,9
135940293,1,2075,10


In [14]:
import json

In [15]:
predictions = reco_hot.groupby("user_id")["item_id"].apply(list).to_dict()

with open(data_base_path / "reco_hot.json", "w") as f:
    json.dump(predictions, f)

In [18]:
reco_cold = user_knn.recommend_cold([10**9], 10)

In [19]:
reco_cold.item_id.to_list()

[10440, 15297, 9728, 13865, 4151, 3734, 2657, 4880, 142, 6809]

In [20]:
with open(data_base_path / "reco_hot.json", "r") as f:
    new = json.load(f)

with open("../data/reco_hot.json", "r") as f:
    old = json.load(f)

In [21]:
diff = 0
for v1, v2 in tqdm(zip(new.values(), old.values())):
    if v1 != v2:
        diff += 1

0it [00:00, ?it/s]

In [22]:
diff

193939